# Step 4 on Colab: main 4x4 training run (back_max 30), snapshots, evaluation

Runtime -> Change runtime type -> **GPU**. Fill in `TOTAL_GPU_HOURS`, then run all cells. Re-run all cells in every new
Colab session until the session log says `finished`.

* **Sessions:** training resumes from the checkpoint on Drive and stops cleanly before `SESSION_HOURS` (the last
  update cycle is not started unless it fits). Each session appends a row to `sessions.csv` on Drive: start/end
  iteration, update_num, deepest training depth (latest greedy check and max so far), iterations/hour, status.
  A row left as `running` by a session that died is closed as `interrupted` by the next session.
* **Snapshots:** at every 10% of `TOTAL_ITRS` the model is copied to `saved_models/cube4/snapshots/itr_<N>/`
  (never overwritten).
* **Curriculum:** none. avi.py draws every training state's scramble depth uniformly from 0..`BACK_MAX` in every
  update cycle; value iteration already learns outward from the goal, because a deep state's target is only useful
  once the states one move closer are learned.
* **Deepest training depth** = largest depth with greedy-check %Solved >= 50%. The greedy check can only solve
  depths <= min(update_num, BACK_MAX - 1), so it also shows how often the target network is updated.
* **Eval cost:** a cube that fails uses the whole node budget. With the defaults, the main evaluation can take a few
  hours on its own; it resumes per section like Step 3, so it can also span sessions.
* If the repository is private, set `REPO_URL` to `https://<token>@github.com/bobcoelau12/Ruba.git` and do not save
  the notebook with the token in it.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/bobcoelau12/Ruba.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/Ruba_step4"
REPO_DIR = "/content/Ruba"

# training budget
TOTAL_GPU_HOURS = None        # <-- fill in (total over all sessions)
ITRS_PER_HOUR = 10000         # measured in Step 3 (back_max 10). Check itrs_per_hour in sessions.csv after the first
                              # session; the deeper greedy check makes back_max 30 somewhat slower.
assert TOTAL_GPU_HOURS is not None, "set TOTAL_GPU_HOURS"
TOTAL_ITRS = int(TOTAL_GPU_HOURS * ITRS_PER_HOUR)
SESSION_HOURS = 3.0           # this session's wall-clock budget; keep it below your Colab session limit
NUM_SNAPSHOTS = 10            # a snapshot at every 10% of TOTAL_ITRS

# training config (avi.py)
BACK_MAX = 30
BATCH_SIZE = 1000
STATES_PER_UPDATE = 200000    # one update cycle = STATES_PER_UPDATE / BATCH_SIZE = 200 iterations, then a checkpoint
NUM_TEST = 1000               # greedy-check states, split over ~30 depths (about 33 per depth)
LOSS_THRESH = 0.05            # target network update threshold (avi.py default, as in Step 3)

# evaluation (node budget per cube, same for both solvers and the baseline)
EVAL_ARGS = ("--targets native4 embedded3 embedded2 --solvers astar beam --depths 1 2 3 4 5 6 7 8 10 12 15 "
             "--num_per_depth 50 --max_nodes 200000 --weight 0.8 --batch_size 100 --beam_width 256 "
             "--beam_max_steps 50")
SNAPSHOT_EVAL_ARGS = EVAL_ARGS.replace("--num_per_depth 50", "--num_per_depth 20")  # cheaper, run for 10 snapshots
RUN_SNAPSHOT_EVAL = False     # set True once training has finished

SAVE_DIR = f"{DRIVE_ROOT}/saved_models"
MODEL_DIR = f"{SAVE_DIR}/cube4"
SESSIONS_CSV = f"{DRIVE_ROOT}/sessions.csv"
RESULTS = f"{DRIVE_ROOT}/results"
print("TOTAL_ITRS =", TOTAL_ITRS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Training and evaluation do not need jumanji (only one test does); skipping it keeps Colab's own JAX untouched.
%cd {REPO_DIR}
!grep -v jumanji requirements.txt > /tmp/requirements_colab.txt && pip install -q -r /tmp/requirements_colab.txt

In [ ]:
# DeepCubeA only uses the GPU when CUDA_VISIBLE_DEVICES is set; otherwise it silently runs on CPU.
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = REPO_DIR
sys.path.insert(0, REPO_DIR)

import torch
from utils import nnet_utils
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
device, devices, on_gpu = nnet_utils.get_device()
print("device:", device, "devices:", devices, "on_gpu:", on_gpu, "-", torch.cuda.get_device_name(0))
assert on_gpu and device.type == "cuda", "DeepCubeA would fall back to CPU"

In [ ]:
# Train this session (resumes from the checkpoint; snapshots and sessions.csv on Drive). Full log: {MODEL_DIR}/output.txt
!python -u scripts/step4_train.py --save_dir {SAVE_DIR} --total_itrs {TOTAL_ITRS} \
    --session_time {int(SESSION_HOURS * 3600)} --num_snapshots {NUM_SNAPSHOTS} --log_csv {SESSIONS_CSV} \
    --back_max {BACK_MAX} --batch_size {BATCH_SIZE} --states_per_update {STATES_PER_UPDATE} \
    --num_test {NUM_TEST} --loss_thresh {LOSS_THRESH} --update_nnet_batch_size 10000

import pandas as pd
display(pd.read_csv(SESSIONS_CSV))
print("snapshots:", sorted(os.listdir(f"{MODEL_DIR}/snapshots")) if os.path.isdir(f"{MODEL_DIR}/snapshots") else [])

In [ ]:
# Deepest depth reached during training, and %Solved per depth, vs iteration (all sessions)
!python scripts/step4_training_depth.py --log {MODEL_DIR}/output.txt --out {RESULTS}/training_depth
from IPython.display import Image, display
for f in ("training_depth.png", "training_solved.png"):
    if os.path.isfile(f"{RESULTS}/training_depth/{f}"):
        display(Image(f"{RESULTS}/training_depth/{f}"))

In [ ]:
# Evaluate the latest snapshot: A* and beam with J4 vs the zero-heuristic baseline, then the gate check.
# Resumable: re-running skips finished sections for the same snapshot and config.
import re
snaps = sorted((int(m.group(1)), d) for d in os.listdir(f"{MODEL_DIR}/snapshots")
               if (m := re.fullmatch(r"itr_(\d+)", d)))
itr, name = snaps[-1]
print("evaluating snapshot", name)
!python scripts/step4_eval.py --model_dir {MODEL_DIR}/snapshots/{name} --out {RESULTS}/eval_{name}.json {EVAL_ARGS}

In [ ]:
# Solve rate vs training iteration over all snapshots (one JSON per snapshot + one baseline JSON; resumable)
if RUN_SNAPSHOT_EVAL:
    !python scripts/step4_eval.py --snapshots_dir {MODEL_DIR}/snapshots --out_dir {RESULTS}/snapshots {SNAPSHOT_EVAL_ARGS}
    display(Image(f"{RESULTS}/snapshots/solve_rate_vs_itr.png"))